# Проверка источников BTCUSDT — 0001

Этот ноутбук повторяет проверки сохранённых HTTP-ответов без сети. Он не обучает модель, не показывает цены финального теста и не подтверждает историческую доступность версий ончейн-метрик. Сырые ответы находятся локально вне Git; описание — в `docs/data_catalog.md`, выводы — в `experiments/0001_btc_data_feasibility.md`.


In [1]:
import hashlib
import json
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "scripts/probe_sources.py").exists():
    root = root.parent
if not (root / "scripts/probe_sources.py").exists():
    raise FileNotFoundError("Откройте ноутбук из корня проекта или notebooks/")
sys.path.insert(0, str(root))
from scripts.probe_sources import audit_klines, audit_metrics, parse_time

summary = json.loads((root / "experiments/0001_source_probe_summary.json").read_text())
run = root / summary["evidence_directory"]
manifest = json.loads((run / "manifest.json").read_text())
assert hashlib.sha256((run / "probe_sources.py").read_bytes()).hexdigest() == summary["script_sha256"]
for entry in manifest:
    if "raw_file" in entry:
        body = (run / entry["raw_file"]).read_bytes()
        assert len(body) == entry["bytes"]
        assert hashlib.sha256(body).hexdigest() == entry["sha256"], entry["id"]
print({"requests": len(manifest), "http_status_counts": summary["http_status_counts"], "raw_integrity": "ok"})

followup = root / summary["followup_evidence_directory"]
for entry in json.loads((followup / "manifest.json").read_text()):
    body = (followup / entry["raw_file"]).read_bytes()
    assert hashlib.sha256(body).hexdigest() == entry["sha256"]
assert json.loads((followup / "checks.json").read_text()) == summary["market_anomaly_followup"]
print({"followup_responses_integrity": "ok"})


{'requests': 63, 'http_status_counts': {'200': 61, '403': 2}, 'raw_integrity': 'ok'}
{'followup_responses_integrity': 'ok'}


## Повторная проверка полноты и допустимости

Сетка полуоткрытая `[start, end)`. Проверяются оба края периода: одинаковое число строк ещё не гарантирует полноту, если есть одновременно повтор и пропуск. Ончейн-данные проверяются по отдельным метрикам, чтобы HTTP 403 одной метрики не скрывал другие.


In [2]:
def payloads(prefix):
    entries = [item for item in manifest if item["id"].startswith(prefix)]
    assert entries, prefix
    return entries, [json.loads((run / item["raw_file"]).read_text()) for item in entries]

checks = []
for key, spec in summary["binance"].items():
    if not isinstance(spec, dict) or "profile" not in spec:
        continue
    entries, bodies = payloads("binance_" + key + "_p")
    rows = [row for body in bodies for row in body]
    actual = audit_klines(rows, parse_time(spec["start"]), parse_time(spec["end_exclusive"]),
                          spec["interval"], parse_time(entries[-1]["received_at"]))
    assert actual == spec["profile"], key
    checks.append({"source": "binance", "sample": key, "rows": actual["rows"],
                   "missing": actual["missing_rows"], "duplicates": actual["duplicate_keys"],
                   "invalid": actual["invalid_rows"] + actual["malformed_rows"]})
for metric, windows in summary["coinmetrics"].items():
    for window, spec in windows.items():
        if spec["status"] != "ok":
            assert spec["http_status"] == 403, (metric, window)
            checks.append({"source": "coinmetrics", "sample": metric + "/" + window,
                           "status": "unavailable", "http_status": spec["http_status"]})
            continue
        entries, bodies = payloads("cm_" + window + "_" + metric + "_p")
        rows = [row for body in bodies for row in body["data"]]
        actual = audit_metrics(rows, parse_time(spec["start"]), parse_time(spec["end_exclusive"]), metric)
        assert actual == spec["profile"], (metric, window)
        checks.append({"source": "coinmetrics", "sample": metric + "/" + window,
                       "rows": actual["rows"], "missing": actual["missing_rows"],
                       "duplicates": actual["duplicate_keys"], "invalid": actual["invalid_rows"],
                       "null_values": actual["missing_metric_values"]})
for check in checks:
    print(check)


{'source': 'binance', 'sample': 'research_1h', 'rows': 35807, 'missing': 1, 'duplicates': 0, 'invalid': 2}
{'source': 'binance', 'sample': 'research_1d', 'rows': 1492, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'old_1h', 'rows': 168, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'old_4h', 'rows': 42, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'old_1d', 'rows': 7, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'old_1m', 'rows': 120, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'recent_1h', 'rows': 336, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'recent_4h', 'rows': 84, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'recent_1d', 'rows': 14, 'missing': 0, 'duplicates': 0, 'invalid': 0}
{'source': 'binance', 'sample': 'recent_1m', 'rows': 120, 'missing': 0, 'duplicates

## Интерпретация

Проверки выше относятся к текущему снимку и сохранённым диапазонам, а не ко всем возможным детализациям. Полная минутная история ещё не проверена. `AssetEODCompletionTime` не доказывает неизменность исторической версии счётчиков. Даже нулевые пропуски и успешные HTTP-ответы не разрешают включать ончейн-фичи в основной бэктест без подтверждения доступности соответствующей версии в прошлом.
